# E13 A/B 试验：功效、效应量与两比例检验

> 覆盖矩阵：试验设计 · Issue #9 扩展 · 见 `docs/ANALYSIS_COVERAGE.md`

## 你将学到
1. **随机分流假设**：可交换性；偷看 / 多重比较如何毁掉结论。
2. **效应量 vs p 值**：提升百分点、相对提升、Cohen 类度量。
3. **样本量 / 功效**：给定基线转化率与 MDE，反推每臂需要多少人。
4. **两比例 z / χ²**：与 `scipy.stats` 对齐；**wanglaoshi 尚无 `ExperimentDesign`（缺口 P0）**。
5. 用 `DataAnalyzer` 做试验日志表的结构体检。

## 包侧优化启示
- 缺统一 `ExperimentDesign.sample_size / two_proportion_test / report`。
- 报告宜输出 `StageResult` 风格（效应、CI、功效、假设检查）。

> **本课增补：** 贝叶斯 A/B（Beta-Binomial）与先验敏感性 / 业务门槛概率。


In [ ]:
QUICK = True
RANDOM_STATE = 42
from wanglaoshi.runtime import ready
ready(seed=RANDOM_STATE)


## 课程研讨笔录（E13）

### 课堂问答题
1. 基线转化率 10%，想检出绝对 +1pp（相对 +10%），α=0.05、功效 80%，每臂大约多少？
2. 为什么「边看边停」会抬高一类错误？
3. 若用户可重复进组，单位是会话还是用户？为何影响方差？

### wanglaoshi 缺口
当前课用 `scipy`/`statsmodels` 手写；目标 API 草图：`wanglaoshi.stats.ExperimentDesign`。


## 1. 合成试验日志 + DataAnalyzer

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from wanglaoshi.analyze import DataAnalyzer

rng = np.random.default_rng(RANDOM_STATE)
n_a, n_b = 8000, 8000
p_a, p_b = 0.10, 0.112  # 绝对 +1.2pp
df = pd.DataFrame({
    'user_id': np.arange(n_a + n_b),
    'variant': ['A'] * n_a + ['B'] * n_b,
    'converted': np.concatenate([
        rng.binomial(1, p_a, n_a),
        rng.binomial(1, p_b, n_b),
    ]),
    'revenue': np.concatenate([
        rng.lognormal(1.0, 0.8, n_a) * rng.binomial(1, p_a, n_a),
        rng.lognormal(1.05, 0.8, n_b) * rng.binomial(1, p_b, n_b),
    ]),
})
print(df.groupby('variant')['converted'].agg(['mean', 'count']))
info = DataAnalyzer(df).get_basic_info()
print({k: info[k] for k in list(info)[:8]})


## 2. 两比例检验与置信区间

In [ ]:
from scipy import stats

def two_prop_z(x1, n1, x2, n2):
    p1, p2 = x1 / n1, x2 / n2
    p = (x1 + x2) / (n1 + n2)
    se = np.sqrt(p * (1 - p) * (1 / n1 + 1 / n2))
    z = (p2 - p1) / se
    pval = 2 * (1 - stats.norm.cdf(abs(z)))
    se_unpooled = np.sqrt(p1 * (1 - p1) / n1 + p2 * (1 - p2) / n2)
    ci = (p2 - p1) + np.array([-1, 1]) * 1.96 * se_unpooled
    return dict(p1=p1, p2=p2, lift_pp=(p2 - p1) * 100, z=z, pvalue=pval, ci95=ci)

g = df.groupby('variant')['converted']
x_a, n_a_ = int(g.get_group('A').sum()), int(g.get_group('A').count())
x_b, n_b_ = int(g.get_group('B').sum()), int(g.get_group('B').count())
res = two_prop_z(x_a, n_a_, x_b, n_b_)
print(res)
# χ² 独立性（2x2）
table = pd.crosstab(df['variant'], df['converted'])
chi2, p_chi, dof, exp = stats.chi2_contingency(table)
print('chi2', chi2, 'p', p_chi, 'dof', dof)


## 3. 样本量与功效（正态近似）

In [ ]:
def n_per_arm(p1, mde_abs, alpha=0.05, power=0.8):
    '''两臂等权、双侧、正态近似下的每臂样本量。'''
    from statsmodels.stats.power import NormalIndPower
    from statsmodels.stats.proportion import proportion_effectsize
    p2 = p1 + mde_abs
    es = proportion_effectsize(p1, p2)
    analysis = NormalIndPower()
    n = analysis.solve_power(effect_size=es, alpha=alpha, power=power, ratio=1.0, alternative='two-sided')
    return int(np.ceil(n)), es

n_need, es = n_per_arm(0.10, 0.01)
print('每臂约需', n_need, 'effectsize(h)=', round(es, 4))
# 当前试验功效（事后）
from statsmodels.stats.proportion import proportion_effectsize
from statsmodels.stats.power import NormalIndPower
es_obs = proportion_effectsize(res['p1'], res['p2'])
pow_obs = NormalIndPower().power(es_obs, nobs1=n_a_, alpha=0.05, ratio=n_b_ / n_a_)
print('观测效应 h=', round(es_obs, 4), '事后功效≈', round(pow_obs, 3))


## 4. 多重比较陷阱（演示）

In [ ]:
# 对 20 个无关指标做检验：在真 H0 下仍会看到「显著」
false_hits = 0
for i in range(20):
    fake = rng.binomial(1, 0.1, size=len(df))
    t = pd.crosstab(df['variant'], fake)
    _, p, _, _ = stats.chi2_contingency(t)
    false_hits += int(p < 0.05)
print('20 次无关检验中 p<0.05 次数:', false_hits, '（期望≈1）')
print('启示：预注册主指标；次指标用 FDR / Bonferroni。')


## 5. 交付摘要

In [ ]:
out = Path('outputs/13_ab_testing')
out.mkdir(parents=True, exist_ok=True)
summary = pd.DataFrame([{
    'p_A': res['p1'], 'p_B': res['p2'], 'lift_pp': res['lift_pp'],
    'z': res['z'], 'pvalue': res['pvalue'],
    'ci_low': res['ci95'][0], 'ci_high': res['ci95'][1],
    'n_per_arm_for_1pp': n_need,
}])
summary.to_csv(out / 'ab_summary.csv', index=False)
print('saved', out / 'ab_summary.csv')
print(summary.T)


## 6. 贝叶斯 A/B：Beta-Binomial 与先验敏感性

> 对照：[PyMC Bayesian A/B Testing](https://www.pymc.io/projects/examples/en/2022.12.0/case_studies/bayesian_ab_testing_introduction.html)

用共轭解析解（不强制 PyMC）：后验为 Beta；估计「B 版转化率超过 A 且提升超过业务门槛 δ」的概率。


In [ ]:
from scipy.stats import beta as beta_dist

# 沿用本节转化计数
x_a, n_a_ = int(g.get_group('A').sum()), int(g.get_group('A').count())
x_b, n_b_ = int(g.get_group('B').sum()), int(g.get_group('B').count())
delta = 0.005  # 业务门槛：绝对提升至少 0.5pp

def posterior_prob_b_beats_a(xa, na, xb, nb, delta=0.0, alpha0=1.0, beta0=1.0, n_mc=50000, rng=None):
    rng = np.random.default_rng(RANDOM_STATE) if rng is None else rng
    pa = rng.beta(alpha0 + xa, beta0 + na - xa, size=n_mc)
    pb = rng.beta(alpha0 + xb, beta0 + nb - xb, size=n_mc)
    return float(np.mean(pb > pa + delta))

priors = {
    'uniform(1,1)': (1.0, 1.0),
    'weak_center_10pct(2,18)': (2.0, 18.0),
    'strong_center_10pct(20,180)': (20.0, 180.0),
}
bayes_rows = []
for name, (a0, b0) in priors.items():
    p = posterior_prob_b_beats_a(x_a, n_a_, x_b, n_b_, delta=delta, alpha0=a0, beta0=b0)
    bayes_rows.append({'prior': name, 'P(pB>pA+delta)': p, 'delta': delta})
bayes_df = pd.DataFrame(bayes_rows)
print(bayes_df)
# 后验均值
print('posterior mean A/B (uniform)',
      (1 + x_a) / (2 + n_a_), (1 + x_b) / (2 + n_b_))
out = Path('outputs/13_ab_testing'); out.mkdir(parents=True, exist_ok=True)
bayes_df.to_csv(out / 'bayesian_ab.csv', index=False)
print('saved bayesian_ab.csv')
try:
    import pymc  # noqa: F401
    print('pymc available: 可用官方 Notebook 做 MCMC；本课解析解已覆盖教学点')
except Exception:
    print('pymc 未安装：使用 Beta-Binomial 共轭解析 + Monte Carlo')


## 附录：深度阅读与作业
- Kohavi et al. *Trustworthy Online Controlled Experiments*
- 作业：把主指标改成 `revenue`（连续），做均值差 t 检验 + bootstrap CI；讨论零膨胀。
- API 地图：`ready` / `DataAnalyzer`；待建 `ExperimentDesign`。
